-Spam Ham Projects Using Word2vec,AvgWord2vec

In [1]:
import pandas as pd
messages = pd.read_csv("SMSSpamCollection.txt", sep='\t', names=["label", "message"])
messages

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [2]:
import gensim
from gensim.models import Word2Vec, KeyedVectors

In [3]:
import gensim.downloader as api

wv = api.load('word2vec-google-news-300')

In [4]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

In [5]:
import re


corpus = []
for i in range(0, len(messages)):
    review = re.sub('[^a-zA-Z]', ' ', messages['message'][i])
    review = review.lower()
    review = review.split()
    
    review = [lemmatizer.lemmatize(word) for word in review]
    review = ' '.join(review)
    corpus.append(review)


In [48]:
[[i,j,k] for i,j,k in zip(list(map(len,corpus)),corpus, messages['message']) if i<1]

[[0, '', '645'], [0, '', ':) '], [0, '', ':-) :-)']]

In [6]:
corpus

['go until jurong point crazy available only in bugis n great world la e buffet cine there got amore wat',
 'ok lar joking wif u oni',
 'free entry in a wkly comp to win fa cup final tkts st may text fa to to receive entry question std txt rate t c s apply over s',
 'u dun say so early hor u c already then say',
 'nah i don t think he go to usf he life around here though',
 'freemsg hey there darling it s been week s now and no word back i d like some fun you up for it still tb ok xxx std chgs to send to rcv',
 'even my brother is not like to speak with me they treat me like aid patent',
 'a per your request melle melle oru minnaminunginte nurungu vettam ha been set a your callertune for all caller press to copy your friend callertune',
 'winner a a valued network customer you have been selected to receivea prize reward to claim call claim code kl valid hour only',
 'had your mobile month or more u r entitled to update to the latest colour mobile with camera for free call the mobile up

In [7]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [8]:
words = []

for sent in corpus:
    sent_token = sent_tokenize(sent)
    for sent in sent_token:
        words.append(simple_preprocess(sent))

In [9]:
words

[['go',
  'until',
  'jurong',
  'point',
  'crazy',
  'available',
  'only',
  'in',
  'bugis',
  'great',
  'world',
  'la',
  'buffet',
  'cine',
  'there',
  'got',
  'amore',
  'wat'],
 ['ok', 'lar', 'joking', 'wif', 'oni'],
 ['free',
  'entry',
  'in',
  'wkly',
  'comp',
  'to',
  'win',
  'fa',
  'cup',
  'final',
  'tkts',
  'st',
  'may',
  'text',
  'fa',
  'to',
  'to',
  'receive',
  'entry',
  'question',
  'std',
  'txt',
  'rate',
  'apply',
  'over'],
 ['dun', 'say', 'so', 'early', 'hor', 'already', 'then', 'say'],
 ['nah',
  'don',
  'think',
  'he',
  'go',
  'to',
  'usf',
  'he',
  'life',
  'around',
  'here',
  'though'],
 ['freemsg',
  'hey',
  'there',
  'darling',
  'it',
  'been',
  'week',
  'now',
  'and',
  'no',
  'word',
  'back',
  'like',
  'some',
  'fun',
  'you',
  'up',
  'for',
  'it',
  'still',
  'tb',
  'ok',
  'xxx',
  'std',
  'chgs',
  'to',
  'send',
  'to',
  'rcv'],
 ['even',
  'my',
  'brother',
  'is',
  'not',
  'like',
  'to',
  'spea

In [10]:
import gensim

### Train Word2Vec from scratch

In [12]:
model = gensim.models.Word2Vec(sentences = words,
                      vector_size = 100) # dimension

In [17]:
model.wv.index_to_key # get all vocabulary

1721

In [15]:
model.corpus_count # vocab size

5569

In [18]:
model.epochs # more epochs more better

5

In [20]:
model.wv.similar_by_word('good')

[('all', 0.9986670017242432),
 ('day', 0.9986067414283752),
 ('morning', 0.9985479712486267),
 ('well', 0.9984474182128906),
 ('about', 0.9983870983123779),
 ('where', 0.9983779788017273),
 ('night', 0.9983338117599487),
 ('not', 0.998330295085907),
 ('happy', 0.9983183741569519),
 ('hope', 0.9982845783233643)]

In [22]:
model.wv['good'].shape

(100,)

In [23]:
# for every word we have 100 dimension, but now for every sentence we need 100 dimention
def avg_word2vec(doc):
    # remove out-of-vocabulary words
    #sent = [word for word in doc if word in model.wv.index_to_key]
    #print(sent)
    
    return np.mean([model.wv[word] for word in doc if word in model.wv.index_to_key],axis=0)
                #or [np.zeros(len(model.wv.index_to_key))], axis=0)
    
    

In [24]:
from tqdm import tqdm

In [25]:
import numpy as np

In [27]:
# apply for every sentence

X = []

for i in tqdm(range(len(words))):
    X.append(avg_word2vec(words[i]))

C:\Users\gurunaml\OneDrive - Firstsource Solutions Ltd\Desktop\ML\ML\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\Users\gurunaml\OneDrive - Firstsource Solutions Ltd\Desktop\ML\ML\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 5569/5569 [00:01<00:00, 4014.68it/s]


In [36]:
X

[array([-0.18320729,  0.22606374,  0.13730791,  0.07832153,  0.09370601,
        -0.49875435,  0.17123593,  0.45369262, -0.25718886, -0.10493071,
        -0.17507143, -0.34172392, -0.05152754,  0.10491817,  0.18904845,
        -0.14222758,  0.13727063, -0.31806764, -0.07211546, -0.51229286,
         0.18993764,  0.09561937,  0.10143686, -0.205259  , -0.02187008,
        -0.00999121, -0.20582968, -0.20427701, -0.25554782,  0.04297148,
         0.3048939 ,  0.00540353,  0.08078805, -0.1814842 , -0.1189192 ,
         0.40122274,  0.04528119, -0.15262349, -0.1009831 , -0.47172895,
         0.08605994, -0.24374971, -0.15801096,  0.02740693,  0.16245846,
         0.0153355 , -0.12073811, -0.04072309,  0.23973401,  0.12968247,
         0.19586249, -0.19348626, -0.07592203,  0.02726306, -0.07454032,
         0.03485602,  0.18238197,  0.06428362, -0.3697967 ,  0.18453763,
        -0.02557527,  0.12914962, -0.022559  , -0.11654995, -0.2798863 ,
         0.27210602,  0.05299265,  0.2359568 , -0.3

In [30]:
len(X)

5569

In [37]:
X_new = np.array(X, dtype=object)

In [38]:
X_new.shape

(5569,)

In [40]:
X_new[0].shape

(100,)

In [41]:
### Dependent feature

y = pd.get_dummies(messages['label'])
y = y.iloc[:,0].values

In [42]:
y.shape

(5572,)

In [45]:
len(messages)

5572

check shape of X_new and y, we lost 3 rows

In [46]:
## Dependent Features
## Output Features
y = messages[list(map(lambda x: len(x)>0 ,corpus))]
y=pd.get_dummies(y['label'])
y=y.iloc[:,0].value s

In [47]:
y.shape

(5569,)

each row of X needs to be added as separate row in df

In [50]:
X[0].shape

(100,)

In [51]:
X[0].reshape(1, -1).shape # 1 row 100 columns

(1, 100)

In [55]:
df = pd.DataFrame()

for i in range(0, len(X)):
    df = df._append(pd.DataFrame(X[i].reshape(1, -1)), ignore_index=True)

C:\Users\gurunaml\AppData\Local\Temp\ipykernel_13084\3814333658.py:4: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = df._append(pd.DataFrame(X[i].reshape(1, -1)), ignore_index=True)


In [56]:
df.head()

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-0.183207,0.226064,0.137308,0.078322,0.093706,-0.498754,0.171236,0.453693,-0.257189,-0.104931,...,0.361480,0.156610,0.024264,0.039112,0.434691,0.172127,0.168007,-0.191290,0.147307,-0.004926
1,-0.173579,0.199607,0.117462,0.068819,0.087206,-0.439673,0.137156,0.405087,-0.227066,-0.085834,...,0.325453,0.130163,0.014887,0.026085,0.373003,0.145411,0.146817,-0.179691,0.136195,-0.011011
2,-0.197848,0.245462,0.149075,0.091498,0.088515,-0.541838,0.172655,0.454074,-0.274453,-0.133805,...,0.356917,0.155649,0.017564,0.017077,0.453049,0.168805,0.116345,-0.229300,0.172692,0.013706
3,-0.257682,0.305349,0.179761,0.110298,0.124441,-0.678546,0.223792,0.620741,-0.352357,-0.134343,...,0.496475,0.208206,0.027948,0.057746,0.577074,0.233885,0.241660,-0.266164,0.201666,-0.015437
4,-0.222049,0.254486,0.158558,0.091603,0.111507,-0.576002,0.190423,0.532158,-0.303428,-0.121491,...,0.424203,0.178523,0.030255,0.051818,0.494290,0.202869,0.198206,-0.235866,0.166261,-0.014224


In [63]:
df['Output'] = y

In [64]:
df.head()

,0,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,Output
0,-0.183207,0.226064,0.137308,0.078322,0.093706,-0.498754,0.171236,0.453693,-0.257189,-0.104931,...,0.156610,0.024264,0.039112,0.434691,0.172127,0.168007,-0.191290,0.147307,-0.004926,True
1,-0.173579,0.199607,0.117462,0.068819,0.087206,-0.439673,0.137156,0.405087,-0.227066,-0.085834,...,0.130163,0.014887,0.026085,0.373003,0.145411,0.146817,-0.179691,0.136195,-0.011011,True
2,-0.197848,0.245462,0.149075,0.091498,0.088515,-0.541838,0.172655,0.454074,-0.274453,-0.133805,...,0.155649,0.017564,0.017077,0.453049,0.168805,0.116345,-0.229300,0.172692,0.013706,False
3,-0.257682,0.305349,0.179761,0.110298,0.124441,-0.678546,0.223792,0.620741,-0.352357,-0.134343,...,0.208206,0.027948,0.057746,0.577074,0.233885,0.241660,-0.266164,0.201666,-0.015437,True
4,-0.222049,0.254486,0.158558,0.091603,0.111507,-0.576002,0.190423,0.532158,-0.303428,-0.121491,...,0.178523,0.030255,0.051818,0.494290,0.202869,0.198206,-0.235866,0.166261,-0.014224,True


In [65]:
df.dropna(axis=0, inplace=True)

In [67]:
## Independent Feature

X = df.drop("Output", axis=1)
y = df["Output"]

In [68]:
## Train Test Split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20)

In [69]:
from sklearn.ensemble import RandomForestClassifier

classifier = RandomForestClassifier()

In [66]:
X.isnull().sum()

0         0
1         0
2         0
3         0
4         0
         ..
96        0
97        0
98        0
99        0
Output    0
Length: 101, dtype: int64

In [70]:
classifier.fit(X_train, y_train)

RandomForestClassifier()

In [71]:
y_pred = classifier.predict(X_test)

In [72]:
from sklearn.metrics import accuracy_score,classification_report
print(accuracy_score(y_test,y_pred))

0.9712230215827338


In [73]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

       False       0.94      0.85      0.89       155
        True       0.98      0.99      0.98       957

    accuracy                           0.97      1112
   macro avg       0.96      0.92      0.94      1112
weighted avg       0.97      0.97      0.97      1112

